# Mass-Spring-Damper Forecasting — Simple Physics-Based Approach

This notebook shows how to forecast the position of a mass-spring-damper system
using a physics-based approach. Instead of a generic machine learning model,
we exploit knowledge of the system dynamics to estimate its parameters from
live sensor data and integrate the equations of motion forward in time.

**System dynamics**

The system obeys:

$$m \ddot{x} + c \dot{x} + k x = F(t)$$

where:
- $m = 1.0$ kg — mass (known)
- $c$ — damping coefficient (unknown, to be estimated)
- $k$ — spring stiffness (unknown, possibly degraded by a fault)
- $F(t) = 0.5 \sin(2\pi t)$ — external harmonic forcing (known)

**Strategy**

1. Collect a window of live position observations.
2. Estimate $k$ and $c$ from the data using least squares.
3. Integrate the equation of motion forward to predict future positions.
4. Submit the forecast.

## Installation — run this cell once

In [ ]:
%pip install "epic-elios-client[notebook]" matplotlib numpy

## Configure the Client

In [ ]:
from epic_client import EPICClient

SERVER_URL = "https://epic.elioslab.net"

# Sampling rate of the contest — check the contest details and set this accordingly.
SAMPLING_RATE_HZ = 1.0
DT = 1.0 / SAMPLING_RATE_HZ

# Mass is known from the system specification.
MASS = 1.0

client = EPICClient(SERVER_URL)

## 1. Log In

In [ ]:
client.login("your-username", "your-password")

## 2. Browse Available Contests

Look for a contest based on the `mass_spring_damper` twin with status `ACTIVE`.

In [ ]:
import pandas as pd

contests = client.list_contests()
contests_df = pd.DataFrame(contests)
contests_df[["contest_id", "name", "status", "twin_id", "sampling_rate_hz", "start_date", "end_date"]]

Copy a `contest_id` from the table above, paste it here, and update `SAMPLING_RATE_HZ` to match.

In [ ]:
CONTEST_ID = "replace-with-contest-id"
SAMPLING_RATE_HZ = 1.0   # update to match the contest
DT = 1.0 / SAMPLING_RATE_HZ

## 3. Register for the Contest

In [ ]:
client.register(CONTEST_ID)

## 4. Collect Live Observations

We collect a window of observations long enough to estimate the system parameters reliably.
At least 30 seconds is recommended — more data gives a better fit.

In [ ]:
observations = await client.collect(CONTEST_ID, duration_seconds=60, csv_path="msd_observations.csv")
print(f"Collected {len(observations)} observations")

## 5. Build the Dataset

In [ ]:
import numpy as np

rows = []
for obs in observations:
    rows.append({
        "sequence_id": obs["sequence_id"],
        "timestamp": obs["timestamp"],
        **obs["sensors"],
    })

df = pd.DataFrame(rows)
df["t"] = (df["sequence_id"] - df["sequence_id"].iloc[0]) * DT
df.head()

## 6. Plot the Position Signal

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 4))
plt.plot(df["t"], df["position"], linewidth=1.2, label="position")
plt.xlabel("Time (s)")
plt.ylabel("Position (m)")
plt.title("Live position stream — Mass-Spring-Damper")
plt.legend()
plt.tight_layout()
plt.show()

## 7. Estimate System Parameters

We discretise the equation of motion:

$$m a_i + c v_i + k x_i = F(t_i)$$

and rearrange:

$$k x_i + c v_i = F(t_i) - m a_i$$

Velocity and acceleration are approximated by central finite differences:

$$v_i = \frac{x_{i+1} - x_{i-1}}{2 \Delta t}, \quad a_i = \frac{x_{i+1} - 2x_i + x_{i-1}}{\Delta t^2}$$

The system $[x_i, v_i] [k, c]^T = b_i$ is solved in the least-squares sense.

In [ ]:
x = df["position"].values
t = df["t"].values

# Central finite differences (exclude first and last points)
v = (x[2:] - x[:-2]) / (2 * DT)
a = (x[2:] - 2 * x[1:-1] + x[:-2]) / (DT ** 2)
x_mid = x[1:-1]
t_mid = t[1:-1]

# External forcing F(t) = 0.5 * sin(2π * t)
F = 0.5 * np.sin(2 * np.pi * t_mid)

# Right-hand side: F(t) - m*a
b = F - MASS * a

# Design matrix [x, v]
A = np.column_stack([x_mid, v])

# Least squares: solve for [k, c]
params, residuals, _, _ = np.linalg.lstsq(A, b, rcond=None)
k_est, c_est = params

print(f"Estimated stiffness  k = {k_est:.4f}  (nominal: 10.0)")
print(f"Estimated damping    c = {c_est:.4f}  (nominal:  0.5)")

## 8. Validate the Fit

We reconstruct the acceleration from the estimated parameters and compare it
against the finite-difference acceleration. A good fit confirms the model is valid.

In [ ]:
a_pred = (F - k_est * x_mid - c_est * v) / MASS

plt.figure(figsize=(12, 4))
plt.plot(t_mid, a, label="measured (finite diff)", linewidth=1.2)
plt.plot(t_mid, a_pred, label="model fit", linewidth=1.2, linestyle="--")
plt.xlabel("Time (s)")
plt.ylabel("Acceleration (m/s²)")
plt.title("Parameter fit validation")
plt.legend()
plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((a - a_pred) ** 2))
print(f"Fit RMSE (acceleration): {rmse:.6f} m/s²")

## 9. Predict Future Positions

We integrate the equation of motion forward from the last observed state
using the 4th-order Runge-Kutta method.

The state vector is $[x, v]$ and the dynamics are:

$$\dot{x} = v, \quad \dot{v} = \frac{F(t) - c\,v - k\,x}{m}$$

In [ ]:
def msd_rhs(t, x, v, k, c, m):
    F = 0.5 * np.sin(2 * np.pi * t)
    dxdt = v
    dvdt = (F - c * v - k * x) / m
    return dxdt, dvdt


def rk4_step(t, x, v, dt, k, c, m):
    k1x, k1v = msd_rhs(t,        x,          v,          k, c, m)
    k2x, k2v = msd_rhs(t + dt/2, x + dt/2*k1x, v + dt/2*k1v, k, c, m)
    k3x, k3v = msd_rhs(t + dt/2, x + dt/2*k2x, v + dt/2*k2v, k, c, m)
    k4x, k4v = msd_rhs(t + dt,   x + dt*k3x,   v + dt*k3v,   k, c, m)
    x_new = x + (dt / 6) * (k1x + 2*k2x + 2*k3x + k4x)
    v_new = v + (dt / 6) * (k1v + 2*k2v + 2*k3v + k4v)
    return x_new, v_new


# Initial state from last two observations
x0 = x[-1]
v0 = (x[-1] - x[-2]) / DT
t0 = t[-1]

# Forecast horizons (in steps)
HORIZONS = [1, 5, 10]
max_horizon = max(HORIZONS)

# Integrate forward step by step
x_curr, v_curr, t_curr = x0, v0, t0
forecast = {}
for step in range(1, max_horizon + 1):
    x_curr, v_curr = rk4_step(t_curr, x_curr, v_curr, DT, k_est, c_est, MASS)
    t_curr += DT
    if step in HORIZONS:
        forecast[step] = x_curr

for h, val in forecast.items():
    print(f"horizon_{h:2d}  →  position = {val:.6f} m")

## 10. Build and Submit the Prediction

In [ ]:
prediction_from_sequence = int(df.iloc[-1]["sequence_id"])

payload = {
    "forecast": {
        f"horizon_{h}": {"position": float(forecast[h])}
        for h in HORIZONS
    }
}

print("Payload:", payload)

submission = client.submit(
    contest_id=CONTEST_ID,
    task_id="forecasting",
    prediction_from_sequence=prediction_from_sequence,
    payload=payload,
)
print("Submission:", submission)

## 11. Check Scores and Leaderboard

In [ ]:
scores = client.get_scores(CONTEST_ID)
leaderboard = client.get_leaderboard(CONTEST_ID)

scores, leaderboard

## Next Steps

This approach assumes the system parameters are constant over the observation
window. Under the `reduced_stiffness` fault, $k$ drifts over time — the
estimated $k$ will be an average over the window.

Possible improvements:

- **Sliding window estimation** — re-estimate $k$ and $c$ using only the
  most recent observations to track parameter drift.
- **Recursive least squares** — update parameter estimates online at each
  new observation without refitting from scratch.
- **Kalman filter** — jointly estimate state and parameters in a
  statistically optimal way.
- **Data-driven models** — train an LSTM or Transformer on historical
  data collected at various fault severities.